<table>
  <tr>
    <td><div align="left"><font size="30">Advanced topics</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/master/figs/BDSimLogo_NoBackgnd%402x.png" width="300"></td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
# Install bdsim.  In JupyterLite, piplite checks the local wheel
# (built with 'make wheel') before falling back to PyPI.
# In a regular Jupyter server this is a no-op (piplite is not available).
LOCAL = True
try:
    import piplite
    await piplite.install('bdsim')
    LOCAL = False
except ImportError:
    pass  # already installed (classic Jupyter)
try:
    import google.colab
    !pip install bdsim
    LOCAL = False
except:
    pass  # not Colab, assume already installed

In this notebook we will explore some advanced topics, building on the example used in the "Getting started" notebook.

# Script files

Jupyter notebooks are useful for exploring ideas, but there is value in building each of your models as a Python script with extension `.py`.  A number of such models are distributed with `bdsim`.  The example from the "Getting started" notebook is called `eg1.py` in the `examples` folder.

# Options

`bdsim` has a sophisticated system to set run-time options.  Options, in decreasing precedence, are given by the command line, the arguments to `BDSim` and environment variables. 

## Command line options

Script files can be run with many options

``` shell
$ ./eg1.py -h           
usage: eg1.py [-h] [--backend [BACKEND]] [--tiles SPEC] [--shape WIDTHxHEIGHT] [--blocks] [-g | +g] [-a | +a] [-H | +H] [+A | -A] [--no-progress] [--verbose] [--debug [psd]] [--simtime SIMTIME] [--animation-rate HZ]
              [--dt DT] [--atol ATOL] [--rtol RTOL] [--max-step DT] [--method NAME] [--quiet] [-p [FILE]] [-o [FILE]] [-j [FILE]] [--set SETPARAM] [--global SETGLOB]

Block diagram simulation framework

options:
  -h, --help            show this help message and exit
  --backend [BACKEND], -b [BACKEND]
                        matplotlib backend to choose; use with no argument or with 'list'/'help' to print available backends (default: None)
  --tiles SPEC, -t SPEC
                        window tiling as RxC or one of: square, wide, tall (default: None)
  --shape WIDTHxHEIGHT  window size as WxH, defaults to matplotlib default (default: None)
  --blocks              Display blocks at startup (default: False)
  -g, --no-graphics     disable graphic display (default: True)
  +g, --graphics        enable graphic display (default: True)
  -a, --no-animation    do not animate graphics (default: False)
  +a, --animation       animate graphics (implies --graphics) (default: False)
  -H, --no-hold         do not hold graphics in done() (default: True)
  +H, --hold            hold graphics in done() (default: True)
  +A, --altscreen       display plots on second monitor (default: True)
  -A, --no-altscreen    do not display plots on second monitor (default: True)
  --no-progress         do not display simulation progress bar (default: True)
  --verbose, -v         debug flags (default: False)
  --debug [psd], -d [psd]
                        debug flags: p/ropagate, s/tate, d/eriv, i/nteractive (default: )
  --simtime SIMTIME, -S SIMTIME
                        simulation time: T or T,dt (default: None)
  --animation-rate HZ   target interactive update cadence for animation/debugger (default: 20.0)
  --dt DT               output sample interval used to build solve_ivp t_eval (default: None)
  --atol ATOL           absolute tolerance for solve_ivp (default: None)
  --rtol RTOL           relative tolerance for solve_ivp (default: None)
  --max-step DT         maximum solve_ivp integration step (default: None)
  --method NAME         solve_ivp method (eg. RK45, DOP853, Radau, BDF, LSODA) (default: RK45)
  --quiet, -q           suppress reports and progress bar (default: False)
  -p [FILE], --pickle [FILE]
                        output pickled simulation results (default filename: bd.out) (default: None)
  -o [FILE], --out [FILE]
                        [deprecated, use -p/--pickle] output pickled simulation results (default: None)
  -j [FILE], --json [FILE]
                        output simulation results as JSON (default filename: bd.json) (default: None)
  --set SETPARAM, -s SETPARAM
                        override block parameter using block:param=value (default: [])
  --global SETGLOB      override global parameter using var=value (default: [])

Environment variables:
  BDSIM              comma-separated key=value pairs that set option defaults,
                     e.g. BDSIM=graphics=True,hold=True
  BDSIMPATH          colon-separated list of extra paths/packages to search for blocks
  BDSIM_NO_TOOLBOXES set to 1/true/yes/on to skip loading external toolboxes
  BDSIM_DEBUG_LAZY_LOAD  set to any value to trace lazy block-class resolution
  BDSIM_DEBUG_DISCOVERY  set to any value to trace block-package discovery
  ```


For example, you can disable graphics

``` shell
$ ./eg1.py -g
```

turn off all chatter

``` shell
$ ./eg1.py -q
```

save the output to a JSON file
``` shell
$ ./eg1.py -j out.json

$ python -m pickle out.pkl
t      = ndarray:float64 (220,)
x      = ndarray:float64 (220, 1)
xnames = ['plant:x_0'] (list)
y      = ndarray:float64 (220, 2)
ynames = ['demand', 'sum.0'] (list)
```

or to a Python pickle file
``` shell
$ ./eg1.py -p out.pkl

$ python -m pickle out.pkl
t      = ndarray:float64 (123,)
x      = ndarray:float64 (123, 1)
xnames = ['plant:x_0'] (list)
y      = ndarray:float64 (123, 2)
ynames = ['demand', 'sum.0'] (list)
```

or change parameter values
``` shell
$ ./eg1.py --set gain.0:K=20
```
where `gain.0` is the name of the block, and `K` is the name of the parameter.

This is a powerful feature, combined with `-q` and `--json`, that allows you to run parameter sweeps
on models (possibly in parallel) and generate a set of result files for analysis.

## Program options

The state of the options is kept within `sim` which is a `BDSim` instance.  Most options that can be set on the command line, can also be passed to `BDSim`.  

To demonstrate this will first recreate the example `bdsim` model in this notebook

In [ ]:
import bdsim
sim = bdsim.BDSim()
bd = sim.blockdiagram()
demand = bd.STEP(T=1, name="demand")
plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right", movie="my_animation.mp4")
scope[0] = plant
scope[1] = demand
plant[0] = 10 * (demand - plant)
bd.compile()

The complete set of options is

In [ ]:
print(sim)

which is a struct-like object.  You can access or set options using dot notation

In [ ]:
sim.options.quiet

In [ ]:
sim.options.quiet = True

# Animations and movies

`SCOPE` plots normally appear once the simulation has run, but it's possible to animate them so you can see the signals evolving over time.  This can be enabled from the command line

``` shell
$ ./eg1.py --animation
```

or in the code by 

``` python
    sim = BDSim(..., animation=True)
```

or by changing the simulation options before `run(...)`

``` python
    im.options.animation = True
    sim.run(bd, T=5)
```

but unfortunately we can't see any of these animations inside a notebook.

Instead we will tell the scope to save the animation as a video file

In [ ]:
bd = sim.blockdiagram()
demand = bd.STEP(T=1, name="demand")
plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right", movie="my_animation.mp4")
scope[0] = plant
scope[1] = demand
plant[0] = 10 * (demand - plant)
bd.compile()
sim.options.animation = True
out = sim.run(bd, T=5)


And we can display the movie inside a Jupyter cell

In [ ]:
from IPython.display import Video
Video("my_animation.mp4")

# Saving plots

By default the `sim.run(...)` method won't return until all `SCOPE` blocks are closed.

We can change that behaviour by using the `block=False` option

``` python
    out = sim.run(..., block=False)

    # ---- do stuff ---- #

    # process out

    # SCOPE windows are visible but interaction (zoom, scroll, data cursors) is inoperative

    scope.savefig()  # saves as scope.0.pdf  (block's name in PDF format)
    scope.savefig("myplot") # saves as myplot.pdf
    scope.savefig("myplot", format="png") # saves as myplot.png

    sim.done() # blocking call that enables the SCOPE interaction
```

The simulation results are returned as usual, but all the `SCOPE` objects still visible.   The `SCOPE` plots can be saved by calling their `savefig()` method.  `sim.savefigs(bd)` will save all `SCOPE`s in the diagram.

# Converting your model back to a diagram

We can export the block diagram to a `.dot` file, which can be rendered with Graphviz or similar tools.  This can be useful to sanity check that the code you've entered matches the model.  

In [ ]:
bd.dotfile("my_model.dot")

creates a [GraphViz format](https://graphviz.org) `.dot` file which represents your block diagram as a mathematical graph using a simple text file format.

GraphViz comes with converters to render `.dot` files into various formats.  We will render this file into SVG and display it inside the notebook.

In [ ]:

if LOCAL:
    
    !dot -Tsvg my_model.dot -o my_model.svg

    from IPython.display import SVG, display
    display(SVG(filename='my_model.svg'))

# Dynamic block loading

All blocks are classes.  For example, you can see the source code of the `GAIN` block [here](https://github.com/petercorke/bdsim/blob/9603af34d51d3ec4ed98b424013a242a60eda165/bdsim/blocks/functions.py#L250).  The class is actually called `Gain`.  There is a hierarchy of block classes: `Gain` → `FunctionBlock` → `Block`.

When we call

In [ ]:
sim = bdsim.BDSim()

a lot of things happen.  Firstly, as discussed above, it processes all the options. 

Secondly, it "loads" all the Python files that define blocks:

In [ ]:
sim.blocks()

The ones prefixed by `bdsim.blocks` are shipped with the `bdsim` package.  Blocks are also defined in companion packages such as the Robotics Toolbox for Python, Machine Vision Toolbox for Python and Spatial Math -- if you have them installed they will be found and listed.

When we create a new blockdiagram

``` python
    bd = sim.blockdiagram()
```

it turns the `bd` instance into a block "factory".  The `__init__` method of class `Gain` is turned into the method call `bd.GAIN`.  Blocks have a number of other methods like `output`, `deriv` and `next` which are used by the runtime simulation engine to simulate the model.

It is quite straightforward to create your own blocks.  Find an existing block close in function to what you want, clone it and then modify it to suit.